In [1]:
# Setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "preprocessing"))  # the notebook lives in models/
import config
import data_exploration   # 1
import clean_dataset      # 2, 3 (uses light_cleaning and heavy_cleaning)
import quality_report     # 4
import balance_languages  # 5
import split_data         # 6

In [2]:
# 1) Exploration report on the raw data (raw/ is never modified)
data_exploration.explore("raw")


===== ar / raw =====
articles: 35,952 | characters: 176.3M | size: 314.5 MB
length: median 1,708 | p10 263 | p90 11,334 | max 176,008
shorter than 150 chars: 557
problems (articles):
  leftover heading 'مراجع': 21,307
  leftover heading 'وصلات خارجية': 8,365
  leftover heading 'المراجع': 6,363
  leftover heading 'انظر أيضًا': 5,396
  leftover heading 'روابط خارجية': 4,370
  leftover heading 'مصادر': 2,737
  leftover heading 'انظر أيضا': 2,216
  leftover heading 'انظر أيضاً': 1,450
  link lines: 471
  table markup: 326
  leftover heading 'Notes': 2
  leftover heading 'References': 1
most common short lines:
   21,328  مراجع
    8,376  وصلات خارجية
    6,375  المراجع
    5,406  انظر أيضًا
    4,375  روابط خارجية
    3,709  وفيات
    3,616  أشخاص على قيد الحياة
    3,343  مواليد
    2,744  مصادر
    2,647  أحداث
random article: فرديناند الأول ملك بلغاريا
الملك فرديناند الأول (26 فبراير 1861 - 10 سبتمبر 1948)، ملك وقيصر بلغاريا (1908 - 1918) . ولد باسم الأمير فرديناند ماكسيملان كارل ليوبو

In [3]:
# 2) Light cleaning (normalization + basic): raw/ -> datasets/cleaned/
light_stats = clean_dataset.run_light_cleaning()

ar: 35,952 raw -> 32,808 after light cleaning
en: 241,787 raw -> 198,037 after light cleaning


In [4]:
# 3) Heavy cleaning (structural) on datasets/cleaned/, then exact duplicates removed
clean_stats = clean_dataset.run_heavy_cleaning(light_stats)

ar: 32,808 -> 30,733 after heavy cleaning
en: 198,037 -> 184,332 after heavy cleaning


In [5]:
# 4) Quality report: raw vs. cleaned
quality_report.run(clean_stats)


===== ar: raw vs cleaned =====
articles: 35,952 -> 30,733 (85.5% kept)
  removed, too short before cleaning: 557
  removed, too short after basic cleaning: 2,587
  removed, too short after structural cleaning: 2,072
  removed, duplicates: 3
size: 314.5 MB -> 278.9 MB (-11.3%)
problems left: none

--- نظرية القانون: 342 -> 326 characters
BEFORE:
علم القانون والحاجة اليه وخصائصه والتمييز بينه وبين القانون الطبيعي ومدى سلطان الافراد ازاء القاعدة القانونية والتفرقة بين القواعد الآمرة  والقواعد المكملة، ومصادر القواعد القانونية وبيان المصادر الرسمية المنشئة للقاعدة القانونية ومعرفة المصادر المباشرة منها، ثم المصادر التفسيرية وهي القضاء والفقه والعادة أو الممارسة العملية.

مراجع 

قانون

AFTER:
علم القانون والحاجة اليه وخصائصه والتمييز بينه وبين القانون الطبيعي ومدى سلطان الافراد ازاء القاعدة القانونية والتفرقة بين القواعد الآمرة والقواعد المكملة، ومصادر القواعد القانونية وبيان المصادر الرسمية المنشئة للقاعدة القانونية ومعرفة المصادر المباشرة منها، ثم المصادر التفسيرية وهي القضاء والفقه وال

In [6]:
# 5) Give Arabic and English about the same number of tokens; the extra articles are deleted
balance_languages.run()

ar: 30,733 articles, ~47.0M tokens
en: 184,332 articles, ~46.3M tokens


In [7]:
# 6) Split 80/10/10 per language, mix ar + en, and save to datasets/splits/
split_data.run()

ar: train 24,586, val 3,073, test 3,074
en: train 147,465, val 18,433, test 18,434
train: 172,051 articles
val: 21,506 articles
test: 21,508 articles
